In [0]:
# Caminho do volume com os arquivos brutos da PREVIC
caminho_volume = "/Volumes/workspace/default/dados_previdencia"

# Exibe os arquivos existentes no volume
display(dbutils.fs.ls(caminho_volume))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dados_previdencia/DSI_2025.csv,DSI_2025.csv,910736,1788747147000
dbfs:/Volumes/workspace/default/dados_previdencia/EPB_1SEMESTRE_2025.csv,EPB_1SEMESTRE_2025.csv,19115509,1788747155000
dbfs:/Volumes/workspace/default/dados_previdencia/EPB_2SEMESTRE_2025.csv,EPB_2SEMESTRE_2025.csv,19069658,1788747155000


In [0]:
# Caminhos dos arquivos originais da PREVIC
arquivo_dsi = f"{caminho_volume}/DSI_2025.csv"
arquivo_epb_1sem = f"{caminho_volume}/EPB_1SEMESTRE_2025.csv"
arquivo_epb_2sem = f"{caminho_volume}/EPB_2SEMESTRE_2025.csv"

# Leitura dos CSVs sem cabeçalho.
# Na camada Bronze, todas as colunas permanecem como texto,
# preservando o conteúdo disponibilizado pela fonte.
opcoes_csv = {
    "header": "false",
    "sep": ";",
    "encoding": "UTF-8",
    "inferSchema": "false"
}

df_dsi_bronze = spark.read.options(**opcoes_csv).csv(arquivo_dsi)
df_epb_1sem_bronze = spark.read.options(**opcoes_csv).csv(arquivo_epb_1sem)
df_epb_2sem_bronze = spark.read.options(**opcoes_csv).csv(arquivo_epb_2sem)

print("Arquivos carregados com sucesso.")

Arquivos carregados com sucesso.


In [0]:
# Validação estrutural dos arquivos carregados

resumo_arquivos = [
    (
        "DSI_2025",
        df_dsi_bronze.count(),
        len(df_dsi_bronze.columns)
    ),
    (
        "EPB_1SEMESTRE_2025",
        df_epb_1sem_bronze.count(),
        len(df_epb_1sem_bronze.columns)
    ),
    (
        "EPB_2SEMESTRE_2025",
        df_epb_2sem_bronze.count(),
        len(df_epb_2sem_bronze.columns)
    )
]

df_resumo = spark.createDataFrame(
    resumo_arquivos,
    ["arquivo", "quantidade_registros", "quantidade_colunas"]
)

display(df_resumo)

arquivo,quantidade_registros,quantidade_colunas
DSI_2025,9938,11
EPB_1SEMESTRE_2025,166225,16
EPB_2SEMESTRE_2025,165661,16


In [0]:
# Criação dos schemas da Arquitetura Medalhão
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Schemas Bronze, Silver e Gold criados com sucesso.")

Schemas Bronze, Silver e Gold criados com sucesso.


In [0]:
from pyspark.sql.functions import lit, current_timestamp

# Acrescenta metadados técnicos sem modificar os campos originais
def preparar_bronze(df, nome_arquivo):
    return (
        df
        .withColumn("_arquivo_origem", lit(nome_arquivo))
        .withColumn("_data_ingestao", current_timestamp())
    )

# Preparação dos DataFrames Bronze
df_dsi_bronze_final = preparar_bronze(
    df_dsi_bronze,
    "DSI_2025.csv"
)

df_epb_1sem_bronze_final = preparar_bronze(
    df_epb_1sem_bronze,
    "EPB_1SEMESTRE_2025.csv"
)

df_epb_2sem_bronze_final = preparar_bronze(
    df_epb_2sem_bronze,
    "EPB_2SEMESTRE_2025.csv"
)

# Gravação no formato Delta
df_dsi_bronze_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.previc_dsi_2025")

df_epb_1sem_bronze_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.previc_epb_1semestre_2025")

df_epb_2sem_bronze_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.previc_epb_2semestre_2025")

print("Tabelas Delta da camada Bronze criadas com sucesso.")

Tabelas Delta da camada Bronze criadas com sucesso.


In [0]:
# Validação das tabelas persistidas na camada Bronze

tabelas_bronze = [
    "workspace.bronze.previc_dsi_2025",
    "workspace.bronze.previc_epb_1semestre_2025",
    "workspace.bronze.previc_epb_2semestre_2025"
]

resultado_bronze = []

for tabela in tabelas_bronze:
    df_tabela = spark.table(tabela)

    resultado_bronze.append(
        (
            tabela,
            df_tabela.count(),
            len(df_tabela.columns)
        )
    )

df_validacao_bronze = spark.createDataFrame(
    resultado_bronze,
    ["tabela", "quantidade_registros", "quantidade_colunas"]
)

display(df_validacao_bronze)

tabela,quantidade_registros,quantidade_colunas
workspace.bronze.previc_dsi_2025,9938,13
workspace.bronze.previc_epb_1semestre_2025,166225,18
workspace.bronze.previc_epb_2semestre_2025,165661,18


## Resultado parcial da ingestão — PREVIC

Foram ingeridos três arquivos públicos disponibilizados pela PREVIC:

- **DSI 2025:** 9.938 registros e 11 campos de origem.
- **EPB 1º semestre de 2025:** 166.225 registros e 16 campos de origem.
- **EPB 2º semestre de 2025:** 165.661 registros e 16 campos de origem.

Os arquivos foram recebidos sem cabeçalho, com separador ponto e vírgula e campos inicialmente tratados como texto. Essa estratégia evita inferências incorretas e preserva o conteúdo original na camada Bronze.

Foram acrescentados somente os metadados técnicos `_arquivo_origem` e `_data_ingestao`. Após a gravação em formato Delta, as quantidades de registros foram novamente verificadas, não sendo identificada perda de dados durante a ingestão.

A etapa de ingestão da camada Bronze continuará com a incorporação dos dados da previdência complementar aberta, provenientes da SUSEP, e dos indicadores socioeconômicos selecionados do IBGE.

In [0]:
# Caminho do volume com os arquivos brutos
caminho_volume = "/Volumes/workspace/default/dados_previdencia"

# Conferência dos arquivos após a inclusão das bases da SUSEP
arquivos_volume = dbutils.fs.ls(caminho_volume)

print(f"Quantidade total de arquivos no volume: {len(arquivos_volume)}")

display(arquivos_volume)

Quantidade total de arquivos no volume: 15


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dados_previdencia/DSI_2025.csv,DSI_2025.csv,910736,1788747147000
dbfs:/Volumes/workspace/default/dados_previdencia/EPB_1SEMESTRE_2025.csv,EPB_1SEMESTRE_2025.csv,19115509,1788747155000
dbfs:/Volumes/workspace/default/dados_previdencia/EPB_2SEMESTRE_2025.csv,EPB_2SEMESTRE_2025.csv,19069658,1788747155000
dbfs:/Volumes/workspace/default/dados_previdencia/LISTAEMPRESAS.csv,LISTAEMPRESAS.csv,13142,1788751958000
dbfs:/Volumes/workspace/default/dados_previdencia/SES_prov_segprev.csv,SES_prov_segprev.csv,215696,1788751958000
dbfs:/Volumes/workspace/default/dados_previdencia/Ses_Contrib_Benef.csv,Ses_Contrib_Benef.csv,1098073,1788751960000
dbfs:/Volumes/workspace/default/dados_previdencia/Ses_vgbl_fundos.csv,Ses_vgbl_fundos.csv,223451,1788751961000
dbfs:/Volumes/workspace/default/dados_previdencia/Ses_vgbl_resgates.csv,Ses_vgbl_resgates.csv,482448,1788751961000
dbfs:/Volumes/workspace/default/dados_previdencia/ses_pgbl_fundos.csv,ses_pgbl_fundos.csv,255452,1788751958000
dbfs:/Volumes/workspace/default/dados_previdencia/ses_pgbl_resgates.csv,ses_pgbl_resgates.csv,448128,1788751959000


In [0]:
from pyspark.sql.functions import lit, current_timestamp

# Configuração dos arquivos SUSEP:
# nome do arquivo, nome da tabela Bronze e codificação
fontes_susep = [
    ("LISTAEMPRESAS.csv", "susep_lista_empresas", "windows-1252"),
    ("Ses_Contrib_Benef.csv", "susep_contrib_benef", "UTF-8"),
    ("Ses_vgbl_fundos.csv", "susep_vgbl_fundos", "UTF-8"),
    ("Ses_vgbl_resgates.csv", "susep_vgbl_resgates", "UTF-8"),
    ("ses_pgbl_fundos.csv", "susep_pgbl_fundos", "UTF-8"),
    ("ses_pgbl_resgates.csv", "susep_pgbl_resgates", "UTF-8"),
    ("ses_pgbl_uf.csv", "susep_pgbl_uf", "UTF-8"),
    ("ses_prev_trad_resgates.csv", "susep_prev_trad_resgates", "UTF-8"),
    ("ses_prev_uf.csv", "susep_prev_uf", "UTF-8"),
    ("SES_prov_segprev.csv", "susep_prov_segprev", "UTF-8"),
    ("ses_quantprev_benef.csv", "susep_quantprev_benef", "UTF-8"),
    ("ses_quantprev_part.csv", "susep_quantprev_part", "UTF-8")
]

resultado_susep = []

for arquivo, nome_tabela, codificacao in fontes_susep:
    caminho_arquivo = f"{caminho_volume}/{arquivo}"

    # Todos os campos permanecem como texto na Bronze
    df_susep = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", codificacao)
        .option("inferSchema", "false")
        .csv(caminho_arquivo)
        .withColumn("_arquivo_origem", lit(arquivo))
        .withColumn("_data_ingestao", current_timestamp())
    )

    tabela_completa = f"workspace.bronze.{nome_tabela}"

    (
        df_susep.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_completa)
    )

    resultado_susep.append(
        (
            nome_tabela,
            df_susep.count(),
            len(df_susep.columns)
        )
    )

df_resumo_susep = spark.createDataFrame(
    resultado_susep,
    ["tabela", "quantidade_registros", "quantidade_colunas"]
)

display(df_resumo_susep)

tabela,quantidade_registros,quantidade_colunas
susep_lista_empresas,233,5
susep_contrib_benef,28103,7
susep_vgbl_fundos,6161,5
susep_vgbl_resgates,19552,7
susep_pgbl_fundos,7101,5
susep_pgbl_resgates,17908,7
susep_pgbl_uf,128742,11
susep_prev_trad_resgates,20081,6
susep_prev_uf,285670,11
susep_prov_segprev,8189,5


In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit, current_timestamp, col

# Caminho do volume
caminho_volume = "/Volumes/workspace/default/dados_previdencia"

# Cria um esquema genérico para preservar todas as colunas do SIDRA
def criar_esquema_ibge(quantidade_colunas):
    return StructType([
        StructField(f"_c{i}", StringType(), True)
        for i in range(quantidade_colunas)
    ])

# Leitura do arquivo de rendimento: 87 colunas de origem
df_ibge_rendimento_bronze = (
    spark.read
    .option("header", "false")
    .option("sep", ";")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("encoding", "UTF-8")
    .schema(criar_esquema_ibge(87))
    .csv(f"{caminho_volume}/ibge_rendimento_7444_sudeste.csv")
    .withColumn(
        "_arquivo_origem",
        lit("ibge_rendimento_7444_sudeste.csv")
    )
    .withColumn("_data_ingestao", current_timestamp())
)

# Leitura do arquivo de população: 88 colunas de origem
df_ibge_populacao_bronze = (
    spark.read
    .option("header", "false")
    .option("sep", ";")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("encoding", "UTF-8")
    .schema(criar_esquema_ibge(88))
    .csv(f"{caminho_volume}/ibge_populacao_6407_sudeste.csv")
    .withColumn(
        "_arquivo_origem",
        lit("ibge_populacao_6407_sudeste.csv")
    )
    .withColumn("_data_ingestao", current_timestamp())
)

# Gravação como tabelas Delta Bronze
(
    df_ibge_rendimento_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.ibge_rendimento_7444_sudeste")
)

(
    df_ibge_populacao_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.ibge_populacao_6407_sudeste")
)

# Verificação das linhas que representam dados territoriais
quantidade_rendimento = (
    df_ibge_rendimento_bronze
    .filter(col("_c0") == "UF")
    .count()
)

quantidade_populacao = (
    df_ibge_populacao_bronze
    .filter(col("_c0") == "UF")
    .count()
)

print("Tabelas IBGE Bronze criadas com sucesso.")
print(f"Linhas territoriais de rendimento: {quantidade_rendimento}")
print(f"Linhas territoriais de população: {quantidade_populacao}")

Tabelas IBGE Bronze criadas com sucesso.
Linhas territoriais de rendimento: 4
Linhas territoriais de população: 36


In [0]:
# Inventário final das tabelas Delta da camada Bronze
df_tabelas_bronze = spark.sql(
    "SHOW TABLES IN workspace.bronze"
)

quantidade_tabelas = df_tabelas_bronze.count()

print(f"Quantidade de tabelas na camada Bronze: {quantidade_tabelas}")

display(
    df_tabelas_bronze
    .select("database", "tableName", "isTemporary")
    .orderBy("tableName")
)

Quantidade de tabelas na camada Bronze: 17


database,tableName,isTemporary
bronze,ibge_populacao_6407_sudeste,false
bronze,ibge_rendimento_7444_sudeste,false
bronze,previc_dsi_2025,false
bronze,previc_epb_1semestre_2025,false
bronze,previc_epb_2semestre_2025,false
bronze,susep_contrib_benef,false
bronze,susep_lista_empresas,false
bronze,susep_pgbl_fundos,false
bronze,susep_pgbl_resgates,false
bronze,susep_pgbl_uf,false


## Resultado final da camada Bronze

A ingestão da camada Bronze foi concluída com **17 tabelas Delta**, organizadas no schema `workspace.bronze`:

- **PREVIC:** 3 tabelas relacionadas à previdência complementar fechada.
- **SUSEP:** 12 tabelas relacionadas à previdência complementar aberta.
- **IBGE:** 2 tabelas com informações de rendimento e população da Região Sudeste.

### Decisões de ingestão

Os dados foram preservados com o maior nível possível de fidelidade em relação às fontes. Os campos foram inicialmente mantidos como texto, e foram adicionados metadados técnicos para identificar o arquivo de origem e a data de ingestão.

As bases da SUSEP possuem cabeçalho e separador ponto e vírgula, mas apresentam diferentes codificações e valores monetários com vírgula decimal.

Os arquivos da PREVIC foram disponibilizados sem cabeçalho. Por esse motivo, as colunas ainda possuem nomes genéricos e somente serão identificadas e tipadas na camada Silver.

Os arquivos do IBGE possuem títulos, cabeçalhos em múltiplas linhas, notas metodológicas e dados organizados em formato largo. A camada Bronze preservou essa estrutura; a transformação para linhas por ano, sexo, estado e faixa etária será realizada na Silver.

### Controles realizados

- Conferência da quantidade de arquivos no volume.
- Validação das quantidades de registros e colunas.
- Comparação dos registros antes e depois da gravação em Delta.
- Registro do arquivo de origem e da data de ingestão.
- Confirmação de que todas as tabelas são permanentes.
- Uso de sobrescrita controlada para permitir a reexecução sem duplicidade.

A camada Bronze constitui a base bruta e rastreável do pipeline. Nenhuma interpretação analítica ou conversão definitiva de tipos foi aplicada nesta etapa.